In [1]:
import cv2
import numpy as np
import torch
from torchvision.models.detection import fasterrcnn_resnet50_fpn
from torchvision.ops import nms

In [2]:
# Load the Faster R-CNN model
model = fasterrcnn_resnet50_fpn(pretrained=True)
model.eval()

C:\Users\HP\anaconda3\lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
C:\Users\HP\anaconda3\lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=FasterRCNN_ResNet50_FPN_Weights.COCO_V1`. You can also use `weights=FasterRCNN_ResNet50_FPN_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


FasterRCNN(
  (transform): GeneralizedRCNNTransform(
      Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
      Resize(min_size=(800,), max_size=1333, mode='bilinear')
  )
  (backbone): BackboneWithFPN(
    (body): IntermediateLayerGetter(
      (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
      (bn1): FrozenBatchNorm2d(64, eps=0.0)
      (relu): ReLU(inplace=True)
      (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
      (layer1): Sequential(
        (0): Bottleneck(
          (conv1): Conv2d(64, 64, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn1): FrozenBatchNorm2d(64, eps=0.0)
          (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
          (bn2): FrozenBatchNorm2d(64, eps=0.0)
          (conv3): Conv2d(64, 256, kernel_size=(1, 1), stride=(1, 1), bias=False)
          (bn3): FrozenBatchNorm2d(256, eps=0.0)
          (relu): ReLU(

In [4]:
cap = cv2.VideoCapture('C:/Users/HP/Py Code/Neural Network/Pytorch/Object detection/v1.mp4')

In [7]:
# Define the Tracker class to track the detected person objects based on the proximity between center points of bounding boxes
class Tracker:
    def __init__(self):
        # Store the center positions of the objects
        self.center_points = {}
        # Keep the count of the IDs
        # each time a new object id detected, the count will increase by one
        self.id_count = 0

    def update(self, objects_rect):
        # Objects boxes and ids
        objects_bbs_ids = []

        # Get center point of new object
        for rect in objects_rect:
            x, y, w, h = rect
            cx = (x + x + w) // 2
            cy = (y + y + h) // 2

            # Find out if that object was detected already
            same_object_detected = False
            for id, pt in self.center_points.items():
                dist = np.hypot(cx - pt[0], cy - pt[1])

                if dist < 35:
                    self.center_points[id] = (cx, cy)
                    objects_bbs_ids.append([x, y, w, h, id])
                    same_object_detected = True
                    break

            # New object is detected we assign the ID to that object
            if same_object_detected is False:
                self.center_points[self.id_count] = (cx, cy)
                objects_bbs_ids.append([x, y, w, h, self.id_count])
                self.id_count += 1

        # Clean the dictionary by center points to remove IDS not used anymore
        new_center_points = {}
        for obj_bb_id in objects_bbs_ids:
            _, _, _, _, object_id = obj_bb_id
            center = self.center_points[object_id]
            new_center_points[object_id] = center

        # Update dictionary with IDs not used removed
        self.center_points = new_center_points.copy()
        return objects_bbs_ids

# Define the region of interest polygon
area_1 = [(15, 8), (1000, 8), (1000, 500), (15, 500)]

# Initialize the tracker
tracker = Tracker()

# Define a set to store person IDs within the region of interest
area1 = set()

while True:
    ret, frame = cap.read()
    if not ret:
        break

    frame = cv2.resize(frame, (1020, 500))

    # Draw the region of interest polygon
    cv2.polylines(frame, [np.array(area_1, np.int32)], True, (0, 255, 0), 3)

    # Convert frame to torch tensor
    tensor_frame = torch.from_numpy(frame).permute(2, 0, 1).float() / 255.0

    # Perform inference
    with torch.no_grad():
        predictions = model([tensor_frame])

    # Extract bounding boxes of detected 'person' objects
    detected_boxes = predictions[0]['boxes'].cpu().numpy()
    detected_scores = predictions[0]['scores'].cpu().numpy()
    detected_classes = predictions[0]['labels'].cpu().numpy()
    
    # Filter out 'person' class detections with scores above a certain threshold
    person_boxes = detected_boxes[(detected_classes == 1) & (detected_scores > 0.7)]  # 1 corresponds to 'person' class

    # Apply non-maximum suppression (NMS) to remove redundant bounding boxes
    keep = nms(torch.tensor(person_boxes.astype(np.float32)), torch.tensor(detected_scores[(detected_classes == 1) & (detected_scores > 0.7)]), iou_threshold=0.5)
    person_boxes = person_boxes[keep]

    objects_rect = []
    for box in person_boxes:
        x1, y1, x2, y2 = box.astype(int)
        objects_rect.append([x1, y1, x2 - x1, y2 - y1])

    # Update tracker with detected bounding boxes
    boxes_ids = tracker.update(objects_rect)

    area1.clear()
    for box_id in boxes_ids:
        x, y, w, h, id = box_id
        cv2.rectangle(frame, (x, y), (x + w, y + h), (255, 0, 255), 2)
        cv2.putText(frame, str(id), (x, y), cv2.FONT_HERSHEY_PLAIN, 3, (255, 0, 0), 2)

        # Check if the object is within the defined region of interest
        result = cv2.pointPolygonTest(np.array(area_1, np.int32), (int((x + w) / 2), int((y + h) / 2)), False)
        if result > 0:
            area1.add(id)

    p = len(area1)
    print(p)
    cv2.putText(frame, 'count:' + str(p), (20, 30), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)
    if p > 5:
        cv2.putText(frame, 'Overloaded', (20, 60), cv2.FONT_HERSHEY_PLAIN, 3, (0, 255, 0), 2)

    cv2.imshow('FRAME', frame)
    if cv2.waitKey(0) & 0xFF == 27:
        break

cap.release()
cv2.destroyAllWindows()
